# Lab 1 — Python Foundations for Agentic AI
### Intro to Agentic AI — Hands-On Lab (Java → Python bridge)

**Before you start:** make sure you completed the *Environment Setup Guide* (Python + Jupyter in VS Code/Cursor) and can run a code cell below (Shift+Enter).

**Goals for today.** You already know how to program (Java, from Programming 1). This lab is *not* "learn programming from scratch" — it's "learn Python's dialect of the ideas you already know," focused on exactly the constructs you'll lean on all semester when building agents: dynamic typing, lists/dicts, control flow, functions with flexible arguments, JSON, exceptions, and simple classes.

**How to use this notebook.** Read each section, run the example cell, then complete the 🔧 **Try It Yourself** cell right after it. Solutions are hidden below each exercise — try first, peek second.

Run the cell below to check your setup:

In [ ]:
import sys
print("Python version:", sys.version)
print("If you see a version number above (3.10+), you're ready to go!")


---
## 1. Variables & Dynamic Typing

In Java, every variable has a declared, fixed type:

```java
String name = "Aria";
int age = 21;
name = 42; // ❌ compile error
```

In Python, a variable is just a **name bound to an object**. No declaration, no fixed type, and a name can be rebound to a different type entirely (Python won't stop you — but your logic should!):


In [ ]:
name = "Aria"       # str
age = 21             # int
gpa = 3.8             # float
is_enrolled = True    # bool

print(name, age, gpa, is_enrolled)
print(type(name), type(age), type(gpa), type(is_enrolled))

name = 42   # totally legal in Python — name now points to an int
print(type(name))


Notice there's **no `;`** and **no curly braces `{}`**. Python uses **indentation** (4 spaces, be consistent!) to mark blocks of code — you'll see this in every control-flow example below. Get this wrong and you get an `IndentationError`, Python's equivalent of a syntax error.

### 🔧 Try It Yourself
Create three variables: `agent_name` (a string), `max_steps` (an int), `temperature` (a float, e.g. `0.7` — this is a real parameter you'll use to control LLM randomness later in the course). Print all three with a single `print()` call.

In [ ]:
# TODO: create agent_name, max_steps, temperature and print them


<details><summary>💡 Solution (click to expand)</summary>

```python
agent_name = "ResearchBot"
max_steps = 10
temperature = 0.7
print(agent_name, max_steps, temperature)
```
</details>

---
## 2. Core Data Types & f-Strings

Python's built-in types map roughly onto Java's, with two big differences: `str` has no separate "char" type, and there's `None` instead of `null`.

| Java | Python |
|---|---|
| `int`, `long` | `int` (arbitrary precision, no overflow) |
| `double`, `float` | `float` |
| `String` | `str` |
| `boolean` | `bool` (`True` / `False` — capitalized!) |
| `null` | `None` |

**f-strings** are Python's answer to `String.format()` / concatenation — put an `f` before the quote and `{}` around any expression:


In [ ]:
agent_name = "ResearchBot"
steps_taken = 3
max_steps = 10

# Java-style concatenation still works, but is clunky:
print("Agent " + agent_name + " has taken " + str(steps_taken) + " steps.")

# f-strings: cleaner, and can embed expressions directly
print(f"Agent {agent_name} has taken {steps_taken}/{max_steps} steps.")
print(f"Budget remaining: {max_steps - steps_taken} steps ({(steps_taken/max_steps)*100:.1f}% used)")


### 🔧 Try It Yourself
Using an f-string, print a sentence that includes a variable `tool_name = "web_search"` and a variable `latency_ms = 245.678`, showing the latency rounded to **1 decimal place**. (Hint: `{value:.1f}`)

In [ ]:
tool_name = "web_search"
latency_ms = 245.678
# TODO: print an f-string using tool_name and latency_ms rounded to 1 decimal


<details><summary>💡 Solution</summary>

```python
print(f"Tool '{tool_name}' responded in {latency_ms:.1f} ms")
```
</details>

---
## 3. Collections: list, tuple, dict, set

This is the single most important section for the rest of the course — agent code is *full* of lists (conversation history), dicts (JSON-like messages, tool arguments), and occasionally tuples/sets.

| Java | Python | Mutable? | Ordered? |
|---|---|---|---|
| `ArrayList<T>` | `list` | ✅ | ✅ |
| — (closest: a `final` array) | `tuple` | ❌ | ✅ |
| `HashMap<K,V>` | `dict` | ✅ | ✅ (insertion order, since 3.7) |
| `HashSet<T>` | `set` | ✅ | ❌ |

### Lists — like `ArrayList`, but no generics needed and they can hold mixed types

In [ ]:
conversation = ["Hello, agent!", "How can I help?", "What's the weather in Cincinnati?"]

conversation.append("Let me check that for you.")   # like .add() in Java
print(conversation)
print("Number of turns:", len(conversation))          # like .size()
print("First message:", conversation[0])
print("Last message:", conversation[-1])               # negative indexing: no Java equivalent!
print("Slice (last two):", conversation[-2:])           # slicing: no Java equivalent!


### Tuples — an immutable, fixed-size sequence (great for things like coordinates or a fixed `(role, content)` pair)

In [ ]:
message = ("user", "What's the weather in Cincinnati?")
role, content = message   # "unpacking" — assigns both at once
print(role, "->", content)

try:
    message[0] = "assistant"   # tuples are immutable
except TypeError as e:
    print("Can't do that:", e)


### Dicts — like `HashMap`, and Python's closest cousin to JSON (which agent frameworks use constantly for messages and tool calls)

In [ ]:
agent_message = {
    "role": "assistant",
    "content": "The weather in Cincinnati is 78F and sunny.",
    "confidence": 0.92,
    "tool_calls": []
}

print(agent_message["role"])          # like .get("role") in Java, but with [] syntax
print(agent_message.get("model", "unknown"))  # .get with a default — no KeyError if missing

agent_message["timestamp"] = "2026-08-04T10:00:00"   # add a new key, like .put()
for key, value in agent_message.items():               # like .entrySet()
    print(f"  {key}: {value}")


### Sets — like `HashSet`, useful for deduplicating things (e.g. tool names an agent has already tried)

In [ ]:
tools_tried = {"web_search", "calculator", "web_search"}   # duplicate is dropped
print(tools_tried)
tools_tried.add("code_interpreter")
print("calculator" in tools_tried)   # like .contains()


### List & dict comprehensions — Python's most-used shortcut, with no direct Java equivalent (closest is a Stream pipeline)

`[expression for item in iterable if condition]` builds a new list in one line.

In [ ]:
latencies_ms = [120, 340, 95, 610, 88, 275]

# Java: loop + ArrayList.add() in an if-block. Python: one line.
fast_calls = [t for t in latencies_ms if t < 200]
print("Fast calls:", fast_calls)

seconds = [t / 1000 for t in latencies_ms]
print("In seconds:", seconds)

# dict comprehension: build a lookup from two lists
tool_names = ["search", "calc", "browser"]
tool_latencies = [120, 340, 95]
latency_by_tool = {name: lat for name, lat in zip(tool_names, tool_latencies)}
print(latency_by_tool)


### 🔧 Try It Yourself
You're given a list of tool-call dicts. Using a **list comprehension**, build a list of just the `"name"` values for tool calls where `"success"` is `True`.

In [ ]:
tool_calls = [
    {"name": "web_search", "success": True},
    {"name": "calculator", "success": False},
    {"name": "code_interpreter", "success": True},
    {"name": "browser", "success": False},
]

# TODO: build successful_tools using a list comprehension
successful_tools = None
print(successful_tools)


<details><summary>💡 Solution</summary>

```python
successful_tools = [tc["name"] for tc in tool_calls if tc["success"]]
print(successful_tools)  # ['web_search', 'code_interpreter']
```
</details>

---
## 4. Control Flow: if / for / while

Same logic as Java, different punctuation: **no parentheses required, no braces, a colon `:` starts the block, and indentation defines the block.**

In [ ]:
confidence = 0.62

if confidence >= 0.8:
    decision = "answer directly"
elif confidence >= 0.5:
    decision = "answer, but flag for review"
else:
    decision = "ask a clarifying question"

print(decision)

# for loop over a list directly (no index bookkeeping needed, unlike Java's classic for-loop)
for tool in ["search", "calculator", "browser"]:
    print(f"Trying tool: {tool}")

# need the index too? use enumerate() (like Java's IntStream.range, but simpler)
for i, tool in enumerate(["search", "calculator", "browser"]):
    print(f"Step {i}: {tool}")

# range() is Python's classic counting loop
for step in range(3):   # 0, 1, 2 — same semantics as Java's for(int i=0;i<3;i++)
    print("step", step)

# while loop — identical concept to Java
steps_taken = 0
max_steps = 3
while steps_taken < max_steps:
    print(f"Agent step {steps_taken}")
    steps_taken += 1   # no ++ operator in Python!


### 🔧 Try It Yourself
Write a `for` loop over `latencies_ms` (below) that prints `"SLOW: {value}"` for any value over 300, and `"ok: {value}"` otherwise.

In [ ]:
latencies_ms = [120, 340, 95, 610, 88, 275]
# TODO: loop and print SLOW/ok for each value


<details><summary>💡 Solution</summary>

```python
for lat in latencies_ms:
    if lat > 300:
        print(f"SLOW: {lat}")
    else:
        print(f"ok: {lat}")
```
</details>

---
## 5. Functions: `def`, default args, `*args` / `**kwargs`, `lambda`

Functions use `def`, don't require a declared return type, and support **default parameter values** and **keyword arguments** — both far more flexible than Java's method overloading.

In [ ]:
def call_agent(prompt, max_tokens=256, temperature=0.7):
    """Docstring: describes what the function does (Python's version of a Javadoc comment)."""
    return f"[calling model with prompt={prompt!r}, max_tokens={max_tokens}, temperature={temperature}]"

print(call_agent("Summarize this article"))                      # uses both defaults
print(call_agent("Summarize this article", temperature=0.2))     # override one, by name
print(call_agent(prompt="Translate this", max_tokens=64, temperature=0))  # all by keyword


**`*args` and `**kwargs`** let a function accept an arbitrary number of positional or keyword arguments — you'll see this constantly in agent framework code (e.g. a generic `tool(*args, **kwargs)` wrapper):

In [ ]:
def run_tool(tool_name, *args, **kwargs):
    print(f"Running '{tool_name}' with args={args} and kwargs={kwargs}")

run_tool("search", "python tutorials", max_results=5)
run_tool("calculator", 2, 3, operation="add")


**`lambda`** creates a small, unnamed, single-expression function — used heavily as the `key=` argument to functions like `sorted()`, which is exactly how you'll rank agent tool results by score:

In [ ]:
results = [
    {"title": "Doc A", "score": 0.72},
    {"title": "Doc B", "score": 0.95},
    {"title": "Doc C", "score": 0.41},
]

# sorted() with a lambda as the sort key — no Comparator class needed like in Java
ranked = sorted(results, key=lambda r: r["score"], reverse=True)
for r in ranked:
    print(r["title"], r["score"])


### 🔧 Try It Yourself
Write a function `summarize_call(tool_name, *args, retries=0, **kwargs)` that returns a single f-string summarizing all four pieces of information. Call it with at least one positional extra arg and one keyword arg.

In [ ]:
# TODO: define summarize_call and call it


<details><summary>💡 Solution</summary>

```python
def summarize_call(tool_name, *args, retries=0, **kwargs):
    return f"tool={tool_name}, args={args}, retries={retries}, extra={kwargs}"

print(summarize_call("search", "python", "AI", retries=2, max_results=5))
```
</details>

---
## 6. Working with JSON

Agent frameworks talk to LLMs and tools almost entirely in JSON (messages, tool schemas, tool call arguments). Python's `dict`/`list` structures map onto JSON almost 1:1 — the `json` module converts between them and text.

In [ ]:
import json

agent_response = {
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {"name": "get_weather", "arguments": {"city": "Cincinnati", "unit": "F"}}
    ]
}

# dict -> JSON string (what you'd send over an API)
json_text = json.dumps(agent_response, indent=2)
print(json_text)
print(type(json_text))

# JSON string -> dict (what you'd receive from an API)
parsed = json.loads(json_text)
print(parsed["tool_calls"][0]["arguments"]["city"])


### 🔧 Try It Yourself
Given the JSON string below (as if received from an LLM tool-call response), parse it and print the `"city"` argument.

In [ ]:
raw = '{"name": "get_weather", "arguments": {"city": "Boston", "unit": "F"}}'
# TODO: parse raw with json.loads and print arguments -> city


<details><summary>💡 Solution</summary>

```python
parsed = json.loads(raw)
print(parsed["arguments"]["city"])
```
</details>

---
## 7. Error Handling: `try` / `except`

Same idea as Java's `try`/`catch`, different keyword (`except` instead of `catch`) and Python exceptions don't need to be declared or checked at compile time.

In [ ]:
def call_tool(tool_name, args):
    if tool_name == "calculator":
        return args["a"] / args["b"]
    raise ValueError(f"Unknown tool: {tool_name}")

try:
    result = call_tool("calculator", {"a": 10, "b": 0})
except ZeroDivisionError:
    print("Handled: can't divide by zero — returning a fallback value")
    result = None
except ValueError as e:
    print(f"Handled: {e}")
    result = None
else:
    print("No error — result:", result)
finally:
    print("This always runs (cleanup, logging, etc.) — same as Java's finally")


This matters a lot for agents: a tool call, an API call, or a JSON parse can fail at runtime, and a well-behaved agent should catch that and recover (retry, fall back, or tell the user) instead of crashing.

### 🔧 Try It Yourself
Wrap a call to `json.loads("not valid json")` in a `try/except` that catches `json.JSONDecodeError` and prints a friendly message instead of crashing.

In [ ]:
import json
# TODO: try/except around json.loads("not valid json")


<details><summary>💡 Solution</summary>

```python
try:
    json.loads("not valid json")
except json.JSONDecodeError as e:
    print(f"Couldn't parse tool response as JSON: {e}")
```
</details>

---
## 8. Classes & Objects — the Lightweight Version

Python classes look a lot like Java's, minus the boilerplate: no access modifiers by convention (a leading underscore `_name` *signals* "private" but isn't enforced), the constructor is always named `__init__`, and every method's first parameter is explicitly `self` (Python's `this`, but you must write it).

In [ ]:
class SimpleAgent:
    def __init__(self, name, max_steps=5):
        self.name = name              # instance field — like Java's this.name = name
        self.max_steps = max_steps
        self.history = []             # each instance gets its own list

    def receive(self, message):
        self.history.append(message)
        return self._respond(message)

    def _respond(self, message):      # leading underscore = "internal use" convention
        if "weather" in message.lower():
            return "Let me check the weather for you."
        return "I'm not sure how to help with that yet."

    def __repr__(self):               # like Java's toString()
        return f"SimpleAgent(name={self.name!r}, turns={len(self.history)})"


agent = SimpleAgent("ResearchBot")
print(agent.receive("What's the weather today?"))
print(agent.receive("Tell me a joke"))
print(agent)          # uses __repr__
print(agent.history)


### 🔧 Try It Yourself
Add a method `reset(self)` to a copy of `SimpleAgent` that clears `self.history` back to an empty list. Create an agent, send it two messages, call `reset()`, and print `history` to confirm it's empty.

In [ ]:
# TODO: define a class with a reset() method and test it


<details><summary>💡 Solution</summary>

```python
class SimpleAgent:
    def __init__(self, name, max_steps=5):
        self.name = name
        self.max_steps = max_steps
        self.history = []

    def receive(self, message):
        self.history.append(message)
        return self._respond(message)

    def _respond(self, message):
        if "weather" in message.lower():
            return "Let me check the weather for you."
        return "I'm not sure how to help with that yet."

    def reset(self):
        self.history = []

agent = SimpleAgent("ResearchBot")
agent.receive("hi")
agent.receive("weather please")
agent.reset()
print(agent.history)  # []
```
</details>

---
## 9. Modules, `import`, and `pip`

Java's `import com.package.Class;` + Maven/Gradle dependencies map onto Python's `import module` + `pip install package`. The standard library covers a lot (`json`, `math`, `random`, `os`, `datetime`); third-party AI packages (`openai`, `anthropic`, `langchain`, etc.) get installed with `pip`.

In [ ]:
import random
import math
from datetime import datetime

print(random.choice(["search", "calculator", "browser"]))   # pick a random tool
print(math.ceil(4.2))
print(datetime.now().isoformat())

# In a terminal (not a notebook cell) you'd install a third-party package with:
#   pip install openai
# then use it the same way: import openai


---
## 10. Put It All Together — Mini Rule-Based Agent

This mini-exercise combines *everything* above: a class, a dict-based "message" format (like real agent frameworks use), control flow, functions, and error handling. This is a small preview of the kind of code you'll be extending for the rest of the course.

In [ ]:
class RuleBasedAgent:
    """A tiny agent that picks a canned response based on keywords.
    Real agents in this course will replace `_decide` with an LLM call —
    but the surrounding structure (history, tool dispatch, error handling)
    stays exactly the same shape."""

    def __init__(self, name):
        self.name = name
        self.history = []          # list of dicts: {"role": ..., "content": ...}
        self.tools_used = set()    # track which tools have been called

    def chat(self, user_message):
        self.history.append({"role": "user", "content": user_message})
        try:
            reply = self._decide(user_message)
        except Exception as e:
            reply = f"Sorry, something went wrong: {e}"
        self.history.append({"role": "assistant", "content": reply})
        return reply

    def _decide(self, message):
        text = message.lower()
        if "weather" in text:
            self.tools_used.add("weather_tool")
            return self._call_weather_tool()
        elif "calculate" in text or "+" in text:
            self.tools_used.add("calculator")
            return "I'd call a calculator tool here."
        elif text.strip() == "":
            raise ValueError("Empty message")
        else:
            return "I don't have a tool for that yet."

    def _call_weather_tool(self):
        # pretend API call
        return "It's 78F and sunny in Cincinnati."


agent = RuleBasedAgent("Lab1Bot")
for msg in ["What's the weather like?", "calculate 2+2", "tell me a story", ""]:
    print(f"USER: {msg!r}")
    print(f"AGENT: {agent.chat(msg)}")
    print()

print("Tools used this session:", agent.tools_used)
print("Full history (as JSON):")
import json
print(json.dumps(agent.history, indent=2))


---
## Wrap-Up

Today you covered the Python constructs you'll use constantly for the rest of this course:

- Dynamic typing & variables
- Core types and f-strings
- Lists, tuples, dicts, sets, and comprehensions
- `if`/`for`/`while` control flow
- Functions, default args, `*args`/`**kwargs`, `lambda`
- JSON serialization/parsing
- `try`/`except` error handling
- Basic classes and objects
- `import` and `pip`

**Next time:** we'll use these exact building blocks to call a real LLM API, parse its JSON tool-call responses, and wire up your first real (non rule-based) agent.

Save this notebook (`Ctrl+S` / `Cmd+S`) — you'll want to reference it later in the semester.